# Deploy an approved DriftWatch model

Run this notebook from the repository root in SageMaker Studio using the same
SageMaker SDK v3 environment as `Pipeline.ipynb` (plus boto3). Training is separate:
this notebook deploys one existing, approved registry version to a new real-time
endpoint. It does not train, approve, register, or replace models.

Fill in the configuration, then run sections 1–7 in order. Section 5 creates
billable AWS hosting resources; an endpoint incurs charges while idle.
Section 8 is optional cleanup and is disabled by default, including on Run All.

Your notebook credentials need permission to describe model packages, create/describe/tag/delete
hosting resources, invoke the endpoint, and pass the execution role (`iam:PassRole`).
The execution role must trust SageMaker and allow access to the registered model's
S3 artifacts, inference source archive, ECR image, and CloudWatch logs (and KMS keys
if applicable). Instance availability and endpoint quotas depend on your region.

References: [deploy from Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-deploy.html)
and [SageMaker SDK v3 resources](https://sagemaker.readthedocs.io/en/stable/api/generated/sagemaker.core.resources.html).

## 1. Configuration

Copy the **model package ARN** from Registry → DriftWatchChurn → Versions →
your approved version → Details. It ends in `/DriftWatchChurn/1` for version 1;
it is not a model group ARN or an S3 URI. Selection is explicit, never “latest.”

Leave region and role empty to use the ARN's region and the Studio execution role.
A blank endpoint name generates a unique name. `ml.m5.large` and one instance are
an initial CPU configuration, not a sizing guarantee; change them for your quotas
and workload. No separate model artifact URI or inference script is needed.

In [ ]:
MODEL_PACKAGE_ARN = ""  # Required: arn:aws:sagemaker:REGION:ACCOUNT:model-package/DriftWatchChurn/1
AWS_REGION = "eu-west-1"         # Optional; defaults to the model package ARN's region
EXECUTION_ROLE_ARN = "" # Optional in Studio; otherwise supply a SageMaker execution role ARN
ENDPOINT_NAME = ""      # Optional; blank generates driftwatch-churn-<unique suffix>
INSTANCE_TYPE = "ml.m5.large"
INSTANCE_COUNT = 1
WAIT_TIMEOUT_SECONDS = 1800
POLL_SECONDS = 30

## 2. Initialize the AWS session

This cell contacts AWS to identify the caller. Credentials come from Studio or
your configured AWS credential chain; do not paste access keys into this notebook.
Cross-account deployment is outside this notebook's scope.

In [ ]:
import json
import math
import re
import time
import uuid
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError
from sagemaker.core.resources import Endpoint, EndpointConfig, Model
from sagemaker.core.shapes.shapes import ContainerDefinition, ProductionVariant, Tag

arn_match = re.fullmatch(
    r"arn:([^:]+):sagemaker:([^:]+):(\d{12}):model-package/([^/]+)/([1-9]\d*)",
    MODEL_PACKAGE_ARN,
)
if not arn_match:
    raise ValueError("Set MODEL_PACKAGE_ARN to a versioned model package ARN.")
partition, package_region, package_account, package_group, package_version = arn_match.groups()
region = AWS_REGION or package_region
if region != package_region:
    raise ValueError("Deploy in the same region as the registered model package.")
if type(INSTANCE_COUNT) is not int or INSTANCE_COUNT < 1:
    raise ValueError("INSTANCE_COUNT must be a positive integer.")
if not isinstance(INSTANCE_TYPE, str) or not INSTANCE_TYPE.startswith("ml."):
    raise ValueError("Set INSTANCE_TYPE to a SageMaker hosting instance type.")
if not (0 < POLL_SECONDS <= WAIT_TIMEOUT_SECONDS):
    raise ValueError("Require 0 < POLL_SECONDS <= WAIT_TIMEOUT_SECONDS.")

aws_session = boto3.Session(region_name=region)
sm = aws_session.client("sagemaker")
runtime = aws_session.client("sagemaker-runtime")
identity = aws_session.client("sts").get_caller_identity()
if identity["Account"] != package_account:
    raise ValueError("Use credentials in the model package's AWS account.")
role_arn = EXECUTION_ROLE_ARN
if not role_arn:
    from sagemaker.core.helper.session_helper import Session, get_execution_role
    role_arn = get_execution_role(sagemaker_session=Session(boto_session=aws_session))
if not re.fullmatch(rf"arn:{re.escape(partition)}:iam::{package_account}:role/.+", role_arn):
    raise ValueError("Supply an execution role ARN in the model package's account.")

suffix = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
endpoint_name = ENDPOINT_NAME or f"driftwatch-churn-{suffix}"
if not re.fullmatch(r"[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?", endpoint_name):
    raise ValueError("Endpoint name must be 1–63 letters, digits, or internal hyphens.")
# Resource types have separate namespaces; use the same name for easy cleanup.
model_name = endpoint_config_name = endpoint_name
tags_dict = {
    "Project": "DriftWatch",
    "ManagedBy": "DriftWatch-Deployment.ipynb",
    "ModelPackage": MODEL_PACKAGE_ARN,
}
resource_tags = [Tag(key=key, value=value) for key, value in tags_dict.items()]
print(json.dumps({
    "account": identity["Account"], "caller": identity["Arn"], "region": region,
    "model_package": MODEL_PACKAGE_ARN, "execution_role": role_arn,
    "endpoint": endpoint_name, "instance_type": INSTANCE_TYPE, "instance_count": INSTANCE_COUNT,
}, indent=2))

## 3. Verify the selected registry version

This checks the actual deployment approval field (`ModelApprovalStatus`), not
just the audit lifecycle label. It also displays linked evaluation metrics for
review. Approval does not itself establish that the model meets your quality needs.

In [ ]:
def approved_package():
    package = sm.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)
    if package["ModelPackageStatus"] != "Completed":
        raise RuntimeError(f"Registration is not complete: {package['ModelPackageStatus']}")
    if package.get("ModelApprovalStatus") != "Approved":
        raise RuntimeError("The selected model version must have ModelApprovalStatus='Approved'.")
    specification = package.get("InferenceSpecification", {})
    if not specification.get("Containers"):
        raise RuntimeError("The registered version has no inference containers.")
    if "application/json" not in specification.get("SupportedContentTypes", []):
        raise RuntimeError("This notebook's test requires registered JSON input support.")
    if "application/json" not in specification.get("SupportedResponseMIMETypes", []):
        raise RuntimeError("This notebook's test requires registered JSON output support.")
    supported = specification.get("SupportedRealtimeInferenceInstanceTypes", [])
    if supported and INSTANCE_TYPE not in supported:
        raise ValueError(f"Select a registered supported instance type: {supported}")
    return package

package = approved_package()
print(json.dumps({
    "version": package_version, "approval": package["ModelApprovalStatus"],
    "inference": package["InferenceSpecification"],
    "metrics": package.get("ModelMetrics", {}),
}, indent=2, default=str))

## 4. Check resource names

Existing endpoints, configurations, and hosting models are never overwritten or
silently reused. If a previous attempt partially succeeded, use the cleanup section
before retrying, or choose a new endpoint name. After a kernel restart, set
`ENDPOINT_NAME` to the printed name to inspect or clean up the earlier attempt.

In [ ]:
def describe_if_present(describe, **kwargs):
    try:
        return describe(**kwargs)
    except ClientError as error:
        details = error.response.get("Error", {})
        message = details.get("Message", "").lower()
        # SageMaker uses ValidationException for absent named resources.
        if details.get("Code") in {"ValidationException", "ResourceNotFound", "ResourceNotFoundException"} and (
            "could not find" in message or "does not exist" in message or "not found" in message
        ):
            return None
        raise  # Access denied, throttling, and other failures must not look like absence.

def check_names_available():
    checks = [
        ("endpoint", sm.describe_endpoint, {"EndpointName": endpoint_name}),
        ("endpoint configuration", sm.describe_endpoint_config, {"EndpointConfigName": endpoint_config_name}),
        ("hosting model", sm.describe_model, {"ModelName": model_name}),
    ]
    for kind, describe, arguments in checks:
        if describe_if_present(describe, **arguments) is not None:
            raise RuntimeError(f"An existing {kind} uses {endpoint_name!r}. Choose a new name or clean up your earlier deployment.")

check_names_available()
print("All three resource names are available.")

## 5. Create the model, configuration, and endpoint — billable

Running this cell provisions hosting resources. The package reference preserves
the registry's image, model data, and inference environment/source configuration.
No retraining, repackaging, or pipeline execution occurs.

If creation fails, earlier resources may still exist. Inspect the error and use
section 8 to clean up; rerunning this cell refuses existing names.

In [ ]:
# Recheck immediately before creating resources, even if earlier cells ran long ago.
approved_package()
check_names_available()
hosting_model = Model.create(
    model_name=model_name,
    execution_role_arn=role_arn,
    containers=[ContainerDefinition(model_package_name=MODEL_PACKAGE_ARN)],
    tags=resource_tags,
    session=aws_session,
    region=region,
)
print(f"Created hosting model: {model_name}")
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[ProductionVariant(
        variant_name="AllTraffic",
        model_name=model_name,
        initial_instance_count=INSTANCE_COUNT,
        instance_type=INSTANCE_TYPE,
        initial_variant_weight=1.0,
    )],
    tags=resource_tags,
    session=aws_session,
    region=region,
)
print(f"Created endpoint configuration: {endpoint_config_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
    tags=resource_tags,
    session=aws_session,
    region=region,
)
print(f"Endpoint creation requested: {endpoint_name}")

## 6. Wait until ready

You can rerun this cell after a timeout without creating another endpoint.
A timeout does not cancel deployment or delete resources. For container startup
errors, inspect CloudWatch Logs at `/aws/sagemaker/Endpoints/<endpoint name>`.

In [ ]:
deadline = time.monotonic() + WAIT_TIMEOUT_SECONDS
while True:
    description = sm.describe_endpoint(EndpointName=endpoint_name)
    status = description["EndpointStatus"]
    print(f"{endpoint_name}: {status}", flush=True)
    if status == "InService":
        print(f"Ready for predictions. Endpoint ARN: {description['EndpointArn']}")
        break
    if status in {"Failed", "OutOfService", "UpdateRollbackFailed", "Deleting"}:
        raise RuntimeError(f"Endpoint is {status}: {description.get('FailureReason', 'See CloudWatch endpoint logs.')}")
    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError("Endpoint is still provisioning. Rerun this wait cell or inspect it in SageMaker Studio; resources remain active.")
    time.sleep(min(POLL_SECONDS, remaining))

## 7. Test a raw customer prediction

This illustrative IBM customer uses raw column names and values; replace it with
a representative unlabeled record matching the training schema. Do not send an
encoded feature vector or a churn label. Missing required fields and unseen
categorical values are rejected by the bundled preprocessing. This is a serving
smoke test, not a new model quality evaluation.

In [ ]:
sample_customer = {
    "Gender": "Female",
    "Senior Citizen": "No",
    "Partner": "Yes",
    "Dependents": "No",
    "Tenure Months": 1,
    "Phone Service": "No",
    "Multiple Lines": "No phone service",
    "Internet Service": "DSL",
    "Online Security": "No",
    "Online Backup": "Yes",
    "Device Protection": "No",
    "Tech Support": "No",
    "Streaming TV": "No",
    "Streaming Movies": "No",
    "Contract": "Month-to-month",
    "Paperless Billing": "Yes",
    "Payment Method": "Electronic check",
    "Monthly Charges": 29.85,
    "Total Charges": 29.85,
}
if sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"] != "InService":
    raise RuntimeError("Wait for the endpoint to reach InService before invoking it.")
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(sample_customer, allow_nan=False).encode("utf-8"),
)
with response["Body"] as response_body:
    prediction = json.loads(response_body.read())
results = prediction.get("predictions") if isinstance(prediction, dict) else None
if not isinstance(results, list) or len(results) != 1 or not isinstance(results[0], dict):
    raise ValueError(f"Unexpected prediction response: {prediction}")
probability = results[0].get("churn_probability")
label = results[0].get("churn_prediction")
if (type(probability) not in (float, int) or not math.isfinite(probability)
        or not 0 <= probability <= 1 or type(label) is not int or label not in (0, 1)):
    raise ValueError(f"Invalid prediction values: {prediction}")
print(json.dumps(prediction, indent=2))
print("Prediction contract verified. The model uses its saved validation-selected threshold.")

## 8. Optional cleanup — disabled by default

Keep the endpoint running if you need it. To remove this deployment, enter the
exact printed endpoint name below and run this cell. It deletes the endpoint,
waits for deletion, then deletes its configuration and hosting model. Registry
versions and S3 artifacts are preserved.

Cleanup also supports partially created deployments and repeated cleanup attempts.
Tags and resource relationships must match this notebook and selected package;
unrelated resources cause it to stop. After a kernel restart, run sections 1–2
and the function definitions in section 4 with the original explicit endpoint name,
then run cleanup. Section 4 may report that names already exist; that is expected.

In [ ]:
DELETE_ENDPOINT_NAME = ""  # To opt in, replace with the exact endpoint name printed above.

def cleanup_deployment():
    if DELETE_ENDPOINT_NAME != endpoint_name:
        print(f"Cleanup skipped. To delete this deployment, set DELETE_ENDPOINT_NAME = {endpoint_name!r}.")
        return
    endpoint_description = describe_if_present(sm.describe_endpoint, EndpointName=endpoint_name)
    config_description = describe_if_present(sm.describe_endpoint_config, EndpointConfigName=endpoint_config_name)
    model_description = describe_if_present(sm.describe_model, ModelName=model_name)
    # Validate every existing resource before deleting any of them.
    for description, arn_field in [
        (endpoint_description, "EndpointArn"),
        (config_description, "EndpointConfigArn"),
        (model_description, "ModelArn"),
    ]:
        if description is None:
            continue
        arn = description[arn_field]
        actual_tags = {}
        arguments = {"ResourceArn": arn}
        while True:
            page = sm.list_tags(**arguments)
            actual_tags.update({item["Key"]: item["Value"] for item in page["Tags"]})
            if not page.get("NextToken"):
                break
            arguments["NextToken"] = page["NextToken"]
        if any(actual_tags.get(key) != value for key, value in tags_dict.items()):
            raise RuntimeError(f"Cleanup refused: ownership tags do not match for {arn}.")
    if endpoint_description and endpoint_description["EndpointConfigName"] != endpoint_config_name:
        raise RuntimeError("Cleanup refused: endpoint uses a different configuration.")
    if config_description and (
        [variant["ModelName"] for variant in config_description["ProductionVariants"]] != [model_name]
        or config_description.get("ShadowProductionVariants")
    ):
        raise RuntimeError("Cleanup refused: configuration references other models.")
    if model_description:
        containers = model_description.get("Containers") or [model_description.get("PrimaryContainer", {})]
        if len(containers) != 1 or containers[0].get("ModelPackageName") != MODEL_PACKAGE_ARN:
            raise RuntimeError("Cleanup refused: hosting model references a different package.")
    if endpoint_description:
        if endpoint_description["EndpointStatus"] != "Deleting":
            sm.delete_endpoint(EndpointName=endpoint_name)
        deadline = time.monotonic() + WAIT_TIMEOUT_SECONDS
        while describe_if_present(sm.describe_endpoint, EndpointName=endpoint_name) is not None:
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                raise TimeoutError("Endpoint deletion is still pending; rerun cleanup before deleting dependent resources.")
            print(f"Waiting for endpoint deletion: {endpoint_name}", flush=True)
            time.sleep(min(POLL_SECONDS, remaining))
    if config_description:
        sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    if model_description:
        sm.delete_model(ModelName=model_name)
    print("Cleanup complete. Registered package and S3 artifacts were preserved.")

cleanup_deployment()